# Screening Credit Agentic AI — eLO System

**Capstone Project — Data Analytics**

Notebook ini mencakup end-to-end pipeline untuk proyek *Screening Credit Agentic AI*:

1. **Generate dataset sintetis** (7 tabel relasional, ribuan baris, terhubung via NIK)
2. **Join & preprocessing** — gabungkan 7 tabel jadi 1 master table analitik
3. **EDA (Exploratory Data Analysis)** dengan visualisasi interaktif Plotly
4. **Agentic Screening Pipeline** — 7 sub-agent (Identity, Credit History, DHN, Collateral,
   Financial, Cashflow, Risk) yang meniru proses 5C penilaian kredit retail banking
5. **Validasi** hasil pipeline terhadap label asli
6. **Export** dataset hasil scoring untuk dipakai di dashboard Streamlit

> ⚠️ Seluruh data di notebook ini adalah **data sintetis** untuk keperluan
> pembelajaran/demo capstone — bukan data nasabah nyata.

## 0. Setup

In [1]:
!pip install -q plotly

import random
import os
from datetime import date, timedelta

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", 100)


## 1. Generate Dataset Sintetis (7 Tabel)

Kita bangun 7 tabel yang merepresentasikan sumber data nyata yang dipakai RB
(Relationship Banking) untuk screening kredit UMKM: **Dukcapil** (identitas),
**SLIK** (riwayat kredit), **DHN** (daftar hitam), **ATR/BPN** (legalitas
agunan), **laporan keuangan**, dan **rekening/mutasi bank**. Semua tabel
terhubung lewat **NIK**.

`retail_customer_profile` adalah tabel ringkasan pengajuan kredit, dilengkapi
`label` (Diterima/Ditolak) yang dihitung dari skor gabungan 4C — jadi bukan
random, tapi hasil turunan logis dari data di tabel lain (lihat fungsi
`compute_label_score` di bawah).

In [2]:
"""
Screening Credit Agentic AI - Synthetic Dataset Generator
============================================================
Generate 7 tabel relasional untuk training model screening kredit retail
banking (5C: Character, Capacity, Collateral, Condition + Capital/Identity).

Semua tabel terhubung lewat NIK (foreign key), KECUALI retail_customer_profile
yang punya application_id sebagai primary key (1 NIK bisa punya banyak
application_id kalau mengajukan berkali-kali, tapi di sini kita generate
1 aplikasi per NIK dulu untuk versi awal).

Cara pakai di Google Colab:
    1. Copy semua isi file ini ke satu cell
    2. Run
    3. 7 file CSV akan tersimpan di /content/dataset/
       (retail_customer_profile.csv, dukcapil.csv, slik_credit_history.csv,
        dhn.csv, agunan_atr_bpn.csv, laporan_keuangan.csv, bank_account.csv)

PENTING saat load ulang CSV-nya nanti (termasuk di tahap join):
    Selalu paksa NIK dibaca sebagai teks, JANGAN biarkan pandas nebak tipenya,
    kalau tidak, 16 digit NIK bisa kepotong presisinya jadi angka:
        pd.read_csv("dukcapil.csv", dtype={"NIK": str})

Catatan penting: nilai tanah/bangunan per kelurahan di sini adalah ESTIMASI
SINTETIS yang dibuat plausible per tingkatan wilayah (bukan data appraisal
resmi/real) - cukup untuk keperluan training model & demo, BUKAN untuk
keputusan bisnis nyata.
"""

import random
import numpy as np
import pandas as pd
from datetime import date, timedelta
import os

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

N_CUSTOMERS = 3000          # jumlah nasabah/debitur unik
OUT_DIR = "/content/dataset" if os.path.isdir("/content") else "./dataset"
os.makedirs(OUT_DIR, exist_ok=True)

# =========================================================================
# 0. REFERENCE / LOOKUP DATA
# =========================================================================

FIRST_NAMES_M = ["Budi","Agus","Andi","Rizky","Dedi","Hendra","Yusuf","Fajar",
    "Wahyu","Bambang","Eko","Rudi","Slamet","Joko","Hadi","Ahmad","Dimas",
    "Arif","Taufik","Iwan","Gunawan","Sutrisno","Anton","Rian","Doni",
    "Yudi","Fauzi","Irfan","Bayu","Krisna"]
FIRST_NAMES_F = ["Siti","Dewi","Rina","Ani","Wulan","Sri","Yuni","Fitri",
    "Indah","Lestari","Ratna","Maya","Putri","Ika","Novi","Wati","Ayu",
    "Dian","Rita","Nina","Sari","Yanti","Lina","Desi","Tri","Retno",
    "Kartika","Anggi","Melati","Suryani"]
LAST_NAMES = ["Santoso","Wijaya","Kurniawan","Saputra","Setiawan","Pratama",
    "Hidayat","Nugroho","Firmansyah","Susanto","Gunawan","Halim","Wibowo",
    "Permana","Suryadi","Handoko","Kusuma","Rahman","Siregar","Simanjuntak",
    "Tanjung","Lubis","Hutapea","Panjaitan","Situmorang"]

RELIGIONS = ["ISLAM","KRISTEN","KATOLIK","HINDU","BUDDHA","KONGHUCU"]
MARITAL = ["Menikah","Belum Menikah","Cerai Hidup","Cerai Mati"]
EDUCATION = ["SMA/SMK","D3","S1","S2"]
BLOOD_TYPE = ["A","B","AB","O"]

INDUSTRIES = {
    "Perdagangan": ["Distributor Elektronik","Toko Sembako","Grosir Pakaian",
                    "Distributor Bahan Bangunan","Toko Alat Tulis"],
    "Kuliner": ["Restoran","Katering","Warung Makan","Bakery"],
    "Jasa": ["Bengkel","Laundry","Percetakan","Jasa Konstruksi Kecil"],
    "Manufaktur": ["Konveksi","Furniture","Pengolahan Makanan Ringan"],
    "Pertanian": ["Distributor Hasil Tani","Peternakan Ayam"],
    "Transportasi": ["Ekspedisi Kecil","Rental Kendaraan"],
}
# Bobot risiko sektor untuk "Condition" (dipakai nanti di Risk Agent, bukan
# dipakai untuk generate label secara langsung supaya label tidak terlalu bocor)
INDUSTRY_RISK = {
    "Perdagangan": 0.05, "Kuliner": 0.10, "Jasa": 0.05,
    "Manufaktur": 0.08, "Pertanian": 0.15, "Transportasi": 0.12,
}

PROVINCES_CITIES = {
    "DKI Jakarta": ["Jakarta Selatan","Jakarta Pusat","Jakarta Timur","Jakarta Barat","Jakarta Utara"],
    "Jawa Barat": ["Bekasi","Depok","Bogor","Tangerang Selatan"],
    "Banten": ["Tangerang"],
}
REGIONS = ["Region 1","Region 2","Region 3","Region 4"]
BRANCHES = ["KCP Tebet","KCP Kelapa Gading","KCP Bekasi Barat","KCP Depok Margonda",
    "KCP Bogor Baranangsiang","KCP Tangerang BSD","KCP Cikini","KCP Kemang",
    "KCP Pluit","KCP Cibubur"]

# Kelurahan lookup untuk agunan (Jabodetabek) -> (provinsi, kota, kecamatan,
# harga tanah/m2 (juta), harga bangunan/m2 (juta)) - ESTIMASI, bukan data resmi
KELURAHAN_LOOKUP = [
    ("DKI Jakarta","Jakarta Selatan","Tebet","Tebet Timur", 28, 5.5),
    ("DKI Jakarta","Jakarta Selatan","Kebayoran Baru","Gunung",45, 6.0),
    ("DKI Jakarta","Jakarta Selatan","Pancoran","Duren Tiga", 30, 5.5),
    ("DKI Jakarta","Jakarta Pusat","Menteng","Menteng", 55, 6.5),
    ("DKI Jakarta","Jakarta Pusat","Cikini","Cikini", 40, 6.0),
    ("DKI Jakarta","Jakarta Timur","Kramat Jati","Kramat Jati", 18, 4.5),
    ("DKI Jakarta","Jakarta Timur","Cakung","Cakung Barat", 12, 4.0),
    ("DKI Jakarta","Jakarta Barat","Kebon Jeruk","Sukabumi Selatan", 22, 5.0),
    ("DKI Jakarta","Jakarta Barat","Cengkareng","Cengkareng Barat", 16, 4.2),
    ("DKI Jakarta","Jakarta Utara","Kelapa Gading","Kelapa Gading Barat", 25, 5.2),
    ("DKI Jakarta","Jakarta Utara","Pluit","Pluit", 27, 5.3),
    ("Jawa Barat","Bekasi","Bekasi Barat","Bintara", 9, 3.8),
    ("Jawa Barat","Bekasi","Bekasi Timur","Margahayu", 8, 3.6),
    ("Jawa Barat","Depok","Beji","Kemiri Muka", 10, 3.8),
    ("Jawa Barat","Depok","Sukmajaya","Mekarjaya", 8.5, 3.6),
    ("Jawa Barat","Bogor","Bogor Tengah","Paledang", 7, 3.4),
    ("Jawa Barat","Tangerang Selatan","Serpong","Rawa Buntu", 12, 4.0),
    ("Banten","Tangerang","Karawaci","Bojong Jaya", 9, 3.6),
    ("Banten","Tangerang","Cipondoh","Poris Plawad", 7.5, 3.4),
]

ASSET_TYPES = ["Tanah","Rumah","Ruko","Gudang"]
CERT_TYPES = ["SHM","HGB"]
LOAN_TYPES = ["KMK","KI","KPR","KKB","KK"]  # Kredit Modal Kerja, Investasi, Pemilikan Rumah, Kendaraan Bermotor, Konsumtif
COLLECT_MAP = {1:"Lancar", 2:"Dalam Perhatian Khusus (DPK)", 3:"Kurang Lancar",
                4:"Diragukan", 5:"Macet"}
OTHER_BANKS = ["Bank Mandiri","Bank BCA","Bank BRI","Bank BNI","Bank CIMB Niaga",
    "Bank Danamon","Bank Permata","Bank OCBC NISP","Bank Panin","BPR Mitra Usaha"]
DHN_REASONS = ["Tunggakan kredit >90 hari di bank lain","Terlibat kasus fraud dokumen",
    "Kredit macet yang belum diselesaikan","Cek/giro kosong berulang",
    "Laporan pihak ketiga terkait sengketa usaha"]

def random_date(start_year, end_year):
    start = date(start_year, 1, 1)
    end = date(end_year, 8, 22)
    delta = (end - start).days
    return start + timedelta(days=random.randint(0, delta))

# kode wilayah (kab/kota) 6-digit ala Kemendagri - plausible, dipakai konsisten dgn kota di dukcapil
KODE_WILAYAH = {
    "Jakarta Selatan": "317401", "Jakarta Pusat": "317101", "Jakarta Timur": "317501",
    "Jakarta Barat": "317301", "Jakarta Utara": "317201",
    "Bekasi": "327501", "Depok": "327601", "Bogor": "327101",
    "Tangerang Selatan": "367401", "Tangerang": "367101",
}

def gen_nik(kota, tanggal_lahir, gender, idx):
    # NIK 16 digit sesuai standar: kode_wilayah(6) + ddmmyy(6, +40 hari utk perempuan) + urutan(4)
    wilayah = KODE_WILAYAH.get(kota, "310101")
    d = tanggal_lahir.day + (40 if gender == "Perempuan" else 0)
    m, y = tanggal_lahir.month, tanggal_lahir.year % 100
    return f"{wilayah}{d:02d}{m:02d}{y:02d}{idx:04d}"


# =========================================================================
# 1. DUKCAPIL (identitas dasar, mengikuti field KTP)
# =========================================================================
def generate_dukcapil(n):
    rows = []
    for i in range(1, n+1):
        gender = random.choice(["Laki-Laki","Perempuan"])
        fname = random.choice(FIRST_NAMES_M if gender=="Laki-Laki" else FIRST_NAMES_F)
        lname = random.choice(LAST_NAMES)
        nama = f"{fname} {lname}"
        prov = random.choice(list(PROVINCES_CITIES.keys()))
        kota = random.choice(PROVINCES_CITIES[prov])
        tgl_lahir = random_date(1965, 2003)
        nik = gen_nik(kota, tgl_lahir, gender, i)
        rows.append({
            "dukcapil_id": f"DKC{i:06d}",
            "NIK": nik,
            "nama": nama,
            "tempat_lahir": kota,
            "tanggal_lahir": tgl_lahir.isoformat(),
            "jenis_kelamin": gender,
            "golongan_darah": random.choice(BLOOD_TYPE),
            "alamat": f"Jl. {random.choice(LAST_NAMES)} No. {random.randint(1,150)}",
            "rt_rw": f"{random.randint(1,12):03d}/{random.randint(1,10):03d}",
            "kelurahan_desa": random.choice(["Sukamaju","Sukajadi","Cempaka Putih",
                "Kebon Baru","Duren Sawit","Rawa Bunga","Cipete","Bintaro"]),
            "kecamatan": random.choice(["Tebet","Kramat Jati","Cengkareng",
                "Bekasi Timur","Sukmajaya","Serpong"]),
            "kota_kabupaten": kota,
            "provinsi": prov,
            "agama": random.choice(RELIGIONS),
            "status_perkawinan": random.choice(MARITAL),
            "pekerjaan": "Wiraswasta",
            "kewarganegaraan": "WNI",
            "berlaku_hingga": "SEUMUR HIDUP",
        })
    return pd.DataFrame(rows)


# =========================================================================
# 2. AGUNAN / ATR-BPN
# =========================================================================
def generate_agunan(dukcapil_df):
    rows = []
    agunan_lookup = {}  # NIK -> summary dict (dipakai utk customer_profile)
    for i, r in enumerate(dukcapil_df.itertuples(), start=1):
        nik = r.NIK
        prov, kota, kec, kel, harga_tanah, harga_bangunan = random.choice(KELURAHAN_LOOKUP)
        asset_type = random.choices(ASSET_TYPES, weights=[0.25,0.30,0.35,0.10])[0]
        land_area = round(np.random.uniform(60, 400), 1)
        building_area = 0.0 if asset_type == "Tanah" else round(land_area * np.random.uniform(0.5, 1.3), 1)
        # variasi harga per unit +/- 15%
        htn = round(harga_tanah * np.random.uniform(0.85, 1.15), 2)
        hbg = round(harga_bangunan * np.random.uniform(0.85, 1.15), 2)
        nilai_tanah = round(land_area * htn * 1_000_000)
        nilai_bangunan = round(building_area * hbg * 1_000_000)
        total_value = nilai_tanah + nilai_bangunan
        ownership_match = np.random.choice(["Ya","Tidak"], p=[0.94, 0.06])
        row = {
            "atr_bpn_id": f"ATR{i:06d}",
            "NIK": nik,
            "asset_type": asset_type,
            "certificate_type": random.choice(CERT_TYPES),
            "certificate_number": f"{random.randint(10000,99999)}/{kel}",
            "provinsi": prov, "kota": kota, "kecamatan": kec, "kelurahan": kel,
            "land_area_m2": land_area,
            "building_area_m2": building_area,
            "nilai_tanah_per_m2": int(htn * 1_000_000),
            "nilai_bangunan_per_m2": int(hbg * 1_000_000),
            "nilai_tanah_total": nilai_tanah,
            "nilai_bangunan_total": nilai_bangunan,
            "total_collateral_value": total_value,
            "ownership_match": ownership_match,
        }
        rows.append(row)
        agunan_lookup[nik] = row
    return pd.DataFrame(rows), agunan_lookup


# =========================================================================
# 3. SLIK CREDIT HISTORY (1-3 fasilitas kredit per NIK, bisa juga 0)
# =========================================================================
def generate_slik(dukcapil_df):
    rows = []
    slik_summary = {}  # NIK -> {"worst_collect":.., "total_installment":.., "n_loans":..}
    rid = 1
    for r in dukcapil_df.itertuples():
        nik = r.NIK
        n_loans = np.random.choice([0,1,2,3], p=[0.15,0.40,0.30,0.15])
        worst = 1
        total_installment = 0
        for _ in range(n_loans):
            plafond = int(np.random.choice([25,50,75,100,150,200,300,500]) * 1_000_000)
            outstanding = int(plafond * np.random.uniform(0.2, 0.95))
            tenor = int(np.random.choice([12,24,36,48,60]))
            installment = int(plafond / tenor * np.random.uniform(1.02,1.15))
            # sebagian besar Lancar, sebagian kecil bermasalah (realistis)
            collect = np.random.choice([1,2,3,4,5], p=[0.72,0.14,0.07,0.04,0.03])
            worst = max(worst, collect)
            total_installment += installment
            rows.append({
                "slik_record_id": f"SLK{rid:06d}",
                "NIK": nik,
                "inquiry_date": random_date(2024,2026).isoformat(),
                "bank_name": random.choice(OTHER_BANKS),
                "loan_type": random.choice(LOAN_TYPES),
                "plafond": plafond,
                "outstanding_balance": outstanding,
                "installment_amount": installment,
                "tenor_month": tenor,
                "collectability": int(collect),
                "collectability_label": COLLECT_MAP[collect],
            })
            rid += 1
        slik_summary[nik] = {"worst_collect": worst, "total_installment": total_installment, "n_loans": n_loans}
    return pd.DataFrame(rows), slik_summary


# =========================================================================
# 4. DHN (Daftar Hitam Nasional) - korelasi dgn worst SLIK collectability
# =========================================================================
def generate_dhn(dukcapil_df, slik_summary):
    rows = []
    dhn_lookup = {}
    for i, r in enumerate(dukcapil_df.itertuples(), start=1):
        nik = r.NIK
        worst = slik_summary[nik]["worst_collect"]
        # makin buruk kolektibilitas SLIK, makin besar peluang masuk DHN
        p_blacklist = {1:0.01, 2:0.03, 3:0.10, 4:0.25, 5:0.45}[worst]
        status = np.random.choice(["Ya","Tidak"], p=[p_blacklist, 1-p_blacklist])
        reason = random.choice(DHN_REASONS) if status == "Ya" else ""
        row = {
            "dhn_id": f"DHN{i:06d}",
            "NIK": nik,
            "status_dhn": status,
            "alasan": reason,
            "tanggal_input": random_date(2023,2026).isoformat(),
        }
        rows.append(row)
        dhn_lookup[nik] = status
    return pd.DataFrame(rows), dhn_lookup


# =========================================================================
# 5. LAPORAN KEUANGAN (2 tahun: 2024 & 2025)
# =========================================================================
def generate_laporan_keuangan(dukcapil_df):
    rows = []
    fin_summary = {}
    rid = 1
    for r in dukcapil_df.itertuples():
        nik = r.NIK
        revenue_2024 = np.random.lognormal(mean=16.8, sigma=0.6)  # ~ puluhan-ratusan juta s/d miliaran
        growth = np.random.normal(0.12, 0.20)  # rata2 tumbuh 12%, bisa negatif
        revenue_2025 = revenue_2024 * (1 + growth)
        margin = np.clip(np.random.normal(0.11, 0.05), 0.01, 0.35)
        recs = []
        for yr, rev in [(2024, revenue_2024), (2025, revenue_2025)]:
            net_profit = rev * margin * np.random.uniform(0.85,1.15)
            total_asset = rev * np.random.uniform(1.1, 2.0)
            total_liability = total_asset * np.random.uniform(0.2, 0.7)
            op_cf = net_profit * np.random.uniform(0.8, 1.4)
            row = {
                "laporan_id": f"FIN{rid:06d}", "NIK": nik, "year": yr,
                "revenue": int(rev), "net_profit": int(net_profit),
                "total_asset": int(total_asset), "total_liability": int(total_liability),
                "operating_cashflow": int(op_cf),
            }
            rows.append(row); recs.append(row); rid += 1
        fin_summary[nik] = {
            "revenue_growth": growth,
            "latest_revenue": recs[1]["revenue"],
            "latest_net_profit": recs[1]["net_profit"],
            "latest_liability": recs[1]["total_liability"],
        }
    return pd.DataFrame(rows), fin_summary


# =========================================================================
# 6. BANK ACCOUNT (1-2 rekening per NIK, korelasi dgn omset)
# =========================================================================
def generate_bank_account(dukcapil_df, fin_summary):
    rows = []
    cf_summary = {}
    aid = 1
    for r in dukcapil_df.itertuples():
        nik = r.NIK
        n_acc = np.random.choice([1,2], p=[0.65,0.35])
        monthly_rev = fin_summary[nik]["latest_revenue"] / 12
        best_avg_balance = 0
        for _ in range(n_acc):
            avg_credit = monthly_rev * np.random.uniform(0.6, 1.1)
            avg_debit = avg_credit * np.random.uniform(0.7, 0.98)
            avg_balance = max(avg_credit - avg_debit, 0) * np.random.uniform(2, 6)
            best_avg_balance = max(best_avg_balance, avg_balance)
            rows.append({
                "account_id": f"ACC{aid:06d}",
                "NIK": nik,
                "account_number": f"{random.randint(1000000000,9999999999):010d}",
                "bank_name": random.choice(["BNI"] + OTHER_BANKS),
                "account_type": random.choice(["Giro","Tabungan"]),
                "account_status": np.random.choice(["Aktif","Dormant"], p=[0.93,0.07]),
                "opened_date": random_date(2015,2025).isoformat(),
                "average_balance_6m": int(avg_balance),
                "average_monthly_credit": int(avg_credit),
                "average_monthly_debit": int(avg_debit),
                "transaction_frequency_monthly": int(np.random.uniform(20,200)),
                "overdraft_count_6m": int(np.random.choice([0,0,0,1,2,3], p=[0.6,0.15,0.1,0.08,0.04,0.03])),
            })
            aid += 1
        cf_summary[nik] = {"best_avg_balance": best_avg_balance}
    return pd.DataFrame(rows), cf_summary


# =========================================================================
# 7. RETAIL CUSTOMER PROFILE (application) + LABEL diterima/ditolak
# =========================================================================
def compute_label_score(worst_collect, dhn_status, growth, net_profit, dsr,
                          collateral_ratio, industry):
    # semua sub-skor di-normalisasi 0..1, makin tinggi makin baik
    s_character = {1:1.0, 2:0.8, 3:0.5, 4:0.25, 5:0.0}[worst_collect]
    if dhn_status == "Ya":
        s_character = min(s_character, 0.1)
    s_capacity = np.clip(0.5 + growth*1.2, 0, 1) * 0.5 + np.clip(1 - dsr, 0, 1) * 0.5
    s_capacity = np.clip(s_capacity, 0, 1)
    s_collateral = np.clip(collateral_ratio / 1.5, 0, 1)  # LTV>=150% -> skor penuh
    s_condition = 1 - INDUSTRY_RISK.get(industry, 0.1) * 4  # sektor riskier -> skor turun sedikit
    s_condition = np.clip(s_condition, 0, 1)

    # bobot 5C (Character & Capacity paling berat, sesuai praktik umum)
    score = 0.35*s_character + 0.30*s_capacity + 0.20*s_collateral + 0.15*s_condition
    score += np.random.normal(0, 0.05)  # noise supaya tidak terlalu deterministik
    return np.clip(score, 0, 1)

def generate_customer_profile(dukcapil_df, agunan_lookup, slik_summary, dhn_lookup,
                                fin_summary, cf_summary):
    rows = []
    for i, r in enumerate(dukcapil_df.itertuples(), start=1):
        nik = r.NIK
        prov, kota = r.provinsi, r.kota_kabupaten
        legal_entity = random.choice(["PT","CV","UD"])
        industry = random.choice(list(INDUSTRIES.keys()))
        sub_industry = random.choice(INDUSTRIES[industry])
        business_age = int(np.random.uniform(1, 20))
        employee_count = int(np.random.uniform(2, 80))
        monthly_turnover = fin_summary[nik]["latest_revenue"] / 12

        agunan = agunan_lookup[nik]
        loan_requested = int(np.random.choice([50,75,100,150,200,300,500,750,1000]) * 1_000_000)
        loan_requested = min(loan_requested, 10_000_000_000)
        collateral_ratio = round(agunan["total_collateral_value"] / max(loan_requested,1), 2)
        collateral_size_m2 = round(agunan["land_area_m2"] + agunan["building_area_m2"], 1)

        slik = slik_summary[nik]
        # estimasi cicilan baru dari pengajuan ini (asumsi tenor 36 bulan, bunga flat ~12%/th)
        new_installment = loan_requested/36 * 1.12
        dsr = (slik["total_installment"] + new_installment) / max(monthly_turnover, 1)

        score = compute_label_score(
            worst_collect=slik["worst_collect"], dhn_status=dhn_lookup[nik],
            growth=fin_summary[nik]["revenue_growth"], net_profit=fin_summary[nik]["latest_net_profit"],
            dsr=dsr, collateral_ratio=collateral_ratio, industry=industry,
        )
        label = "Diterima" if score >= 0.55 else "Ditolak"

        rows.append({
            "application_id": f"APP{2026}{i:05d}",
            "NIK": nik,
            "cif_number": f"CIF{1000000+i}",
            "application_date": random_date(2025,2026).isoformat(),
            "customer_type": "UMKM",
            "company_name": f"{random.choice(['PT','CV','UD'])} {random.choice(LAST_NAMES)} {random.choice(['Jaya','Makmur','Sejahtera','Abadi','Mandiri'])}",
            "legal_entity": legal_entity,
            "owner_name": r.nama,
            "owner_gender": "L" if r.jenis_kelamin=="Laki-Laki" else "P",
            "owner_age": date.today().year - int(r.tanggal_lahir[:4]),
            "owner_marital_status": r.status_perkawinan,
            "owner_education": random.choice(EDUCATION),
            "province": prov, "city": kota,
            "district": r.kecamatan, "region": random.choice(REGIONS),
            "branch_name": random.choice(BRANCHES),
            "industry": industry, "sub_industry": sub_industry,
            "business_age_year": business_age,
            "employee_count": employee_count,
            "monthly_turnover_est": int(monthly_turnover),
            "transaction_frequency_monthly": int(np.random.uniform(30,200)),
            "loan_requested": loan_requested,
            "collateral_type": agunan["asset_type"],
            "collateral_location": f"{agunan['kelurahan']}, {agunan['kota']}",
            "collateral_province": agunan["provinsi"], "collateral_city": agunan["kota"],
            "collateral_size_m2": collateral_size_m2,
            "collateral_market_value": agunan["total_collateral_value"],
            "collateral_liquidation_value": int(agunan["total_collateral_value"] * 0.8),
            "collateral_ratio": collateral_ratio,
            "certificate_type": agunan["certificate_type"],
            "ownership_match": agunan["ownership_match"],
            "estimated_dsr": round(min(dsr,3.0), 2),
            "eligibility_score": round(float(score), 3),
            "label": label,
        })
    return pd.DataFrame(rows)

In [ ]:
# print(f"Generating {N_CUSTOMERS} customers...")
# dukcapil_df = generate_dukcapil(N_CUSTOMERS)
# agunan_df, agunan_lookup = generate_agunan(dukcapil_df)
# slik_df, slik_summary = generate_slik(dukcapil_df)
# dhn_df, dhn_lookup = generate_dhn(dukcapil_df, slik_summary)
# fin_df, fin_summary = generate_laporan_keuangan(dukcapil_df)
# bank_df, cf_summary = generate_bank_account(dukcapil_df, fin_summary)
# profile_df = generate_customer_profile(dukcapil_df, agunan_lookup, slik_summary,
#                                         dhn_lookup, fin_summary, cf_summary)

# tables = {
#     "retail_customer_profile": profile_df, "dukcapil": dukcapil_df,
#     "slik_credit_history": slik_df, "dhn": dhn_df,
#     "agunan_atr_bpn": agunan_df, "laporan_keuangan": fin_df, "bank_account": bank_df,
# }
# for name, df in tables.items():
#     if "NIK" in df.columns:
#         df["NIK"] = df["NIK"].astype(str)
#     if "account_number" in df.columns:
#         df["account_number"] = df["account_number"].astype(str)
#     df.to_csv(os.path.join(OUT_DIR, f"{name}.csv"), index=False)

# print("Baris per tabel:")
# for name, df in tables.items():
#     print(f"  {name:28s} -> {len(df):6d} baris")

# print("\nDistribusi label (retail_customer_profile):")
# print(profile_df["label"].value_counts(normalize=True).round(3))

Generating 3000 customers...
Baris per tabel:
  retail_customer_profile      ->   3000 baris
  dukcapil                     ->   3000 baris
  slik_credit_history          ->   4416 baris
  dhn                          ->   3000 baris
  agunan_atr_bpn               ->   3000 baris
  laporan_keuangan             ->   6000 baris
  bank_account                 ->   4052 baris

Distribusi label (retail_customer_profile):
label
Diterima    0.848
Ditolak     0.152
Name: proportion, dtype: float64


In [1]:
from pathlib import Path
import pandas as pd

# Path folder raw (karena notebook ada di folder notebooks/)
raw_path = Path("../data/raw")

# Dictionary untuk menyimpan semua DataFrame
dfs = {}

# Load semua file CSV
for file in raw_path.glob("*.csv"):
    df_name = file.stem
    dfs[df_name] = pd.read_csv(file)

    print(f"✅ {df_name}: {dfs[df_name].shape[0]} baris x {dfs[df_name].shape[1]} kolom")

✅ agunan_atr_bpn: 3000 baris x 17 kolom
✅ bank_account: 4052 baris x 12 kolom
✅ dhn: 3000 baris x 5 kolom
✅ dukcapil: 3000 baris x 18 kolom
✅ laporan_keuangan: 6000 baris x 8 kolom
✅ retail_customer_profile: 3000 baris x 37 kolom
✅ slik_credit_history: 4416 baris x 11 kolom


## 2. Join & Preprocessing

Tabel `slik_credit_history`, `bank_account`, dan `laporan_keuangan` bersifat
**1:banyak** terhadap NIK (misal 1 nasabah bisa punya beberapa fasilitas
kredit), jadi kita **agregasi** dulu ke 1 baris per NIK sebelum di-join ke
`retail_customer_profile` (tabel utama, unit analisisnya = 1 pengajuan).

Preprocessing yang dilakukan:
- Isi NaN hasil agregasi SLIK dengan nilai yang bermakna (0 = belum ada
  riwayat, **bukan** disamakan dengan kolektibilitas "Lancar")
- Isi `dhn_alasan` kosong dengan "Tidak Berlaku" (bukan data hilang)
- Konversi tipe tanggal
- Tandai kolom **leakage** (`eligibility_score` — jangan dipakai sebagai fitur
  model) dan kolom **sensitif** (`owner_gender`, `owner_marital_status` — demi
  praktik *fair lending*, sebaiknya tidak dipakai sebagai fitur model meskipun
  secara statistik berkorelasi)

In [4]:
NIK_STR = {"NIK": str}

# Load ulang dari CSV (praktik baik: pastikan pipeline tetap jalan walau
# generate & join dilakukan di sesi/notebook terpisah)
profile  = pd.read_csv(f"{OUT_DIR}/retail_customer_profile.csv", dtype=NIK_STR)
dukcapil = pd.read_csv(f"{OUT_DIR}/dukcapil.csv", dtype=NIK_STR)
slik     = pd.read_csv(f"{OUT_DIR}/slik_credit_history.csv", dtype=NIK_STR)
dhn      = pd.read_csv(f"{OUT_DIR}/dhn.csv", dtype=NIK_STR)
agunan   = pd.read_csv(f"{OUT_DIR}/agunan_atr_bpn.csv", dtype=NIK_STR)
fin      = pd.read_csv(f"{OUT_DIR}/laporan_keuangan.csv", dtype=NIK_STR)
bank     = pd.read_csv(f"{OUT_DIR}/bank_account.csv", dtype={**NIK_STR, "account_number": str})

# --- Agregasi tabel 1:banyak -> 1 baris per NIK ---
slik_agg = (slik.groupby("NIK")
    .agg(slik_n_loans=("slik_record_id","count"),
         slik_worst_collectability=("collectability","max"),
         slik_n_banks=("bank_name","nunique"),
         slik_total_outstanding=("outstanding_balance","sum"),
         slik_total_installment_other=("installment_amount","sum"),
         slik_avg_tenor_month=("tenor_month","mean"))
    .reset_index())
slik_agg["slik_has_macet"] = (slik_agg["slik_worst_collectability"] == 5).astype(int)
slik_agg["slik_has_credit_history"] = 1

bank_agg = (bank.groupby("NIK")
    .agg(bank_n_accounts=("account_id","count"),
         bank_best_avg_balance_6m=("average_balance_6m","max"),
         bank_total_avg_credit=("average_monthly_credit","sum"),
         bank_total_avg_debit=("average_monthly_debit","sum"),
         bank_total_overdraft_6m=("overdraft_count_6m","sum"))
    .reset_index())
bank_agg["bank_any_dormant"] = bank.groupby("NIK")["account_status"] \
    .apply(lambda s: int((s == "Dormant").any())).values

fin_pivot = fin.pivot_table(index="NIK", columns="year",
    values=["revenue","net_profit","total_asset","total_liability","operating_cashflow"])
fin_pivot.columns = [f"{col}_{yr}" for col, yr in fin_pivot.columns]
fin_pivot = fin_pivot.reset_index()
fin_pivot["revenue_growth_pct"] = ((fin_pivot["revenue_2025"] - fin_pivot["revenue_2024"])
                                     / fin_pivot["revenue_2024"]).round(4)
fin_pivot["profit_margin_2025"] = (fin_pivot["net_profit_2025"] / fin_pivot["revenue_2025"]).round(4)
fin_pivot["liability_to_asset_2025"] = (fin_pivot["total_liability_2025"]
                                          / fin_pivot["total_asset_2025"]).round(4)

agunan_extra = agunan[["NIK","kelurahan","land_area_m2","building_area_m2",
                        "nilai_tanah_per_m2","nilai_bangunan_per_m2"]].rename(
    columns={"kelurahan":"agunan_kelurahan"})
dhn_slim = dhn[["NIK","status_dhn","alasan"]].rename(columns={"alasan":"dhn_alasan"})

# --- Join semuanya ke master table ---
master = (profile
    .merge(dhn_slim, on="NIK", how="left")
    .merge(slik_agg, on="NIK", how="left")
    .merge(bank_agg, on="NIK", how="left")
    .merge(fin_pivot, on="NIK", how="left")
    .merge(agunan_extra, on="NIK", how="left"))

# --- Preprocessing ---
slik_num_cols = ["slik_n_loans","slik_worst_collectability","slik_n_banks",
                  "slik_total_outstanding","slik_total_installment_other",
                  "slik_avg_tenor_month","slik_has_macet"]
master[slik_num_cols] = master[slik_num_cols].fillna(0)
master["slik_has_credit_history"] = master["slik_has_credit_history"].fillna(0).astype(int)
master["dhn_alasan"] = master["dhn_alasan"].fillna("Tidak Berlaku")
master["application_date"] = pd.to_datetime(master["application_date"])

master["dsr_capped"] = master["estimated_dsr"].clip(upper=3.0)
master["is_female_owner"] = (master["owner_gender"] == "P").astype(int)
master["has_dhn_flag"] = (master["status_dhn"] == "Ya").astype(int)

missing = master.isna().sum()
missing = missing[missing > 0]
print("Kolom yang masih missing setelah preprocessing:", list(missing.index) if len(missing) else "(tidak ada)")

LEAKAGE_COLS = ["eligibility_score"]
SENSITIVE_COLS = ["owner_gender", "is_female_owner", "owner_marital_status"]
print(f"Kolom leakage (WAJIB drop sblm modeling): {LEAKAGE_COLS}")
print(f"Kolom sensitif (drop dari fitur model, fair-lending): {SENSITIVE_COLS}")

master.to_csv(f"{OUT_DIR}/master_dataset.csv", index=False)
print(f"\nMaster table: {master.shape[0]} baris x {master.shape[1]} kolom")
master.head(3)


Kolom yang masih missing setelah preprocessing: (tidak ada)
Kolom leakage (WAJIB drop sblm modeling): ['eligibility_score']
Kolom sensitif (drop dari fitur model, fair-lending): ['owner_gender', 'is_female_owner', 'owner_marital_status']

Master table: 3000 baris x 74 kolom


,application_id,NIK,cif_number,application_date,customer_type,company_name,legal_entity,owner_name,owner_gender,owner_age,owner_marital_status,owner_education,province,city,district,region,branch_name,industry,sub_industry,business_age_year,employee_count,monthly_turnover_est,transaction_frequency_monthly,loan_requested,collateral_type,collateral_location,collateral_province,collateral_city,collateral_size_m2,collateral_market_value,collateral_liquidation_value,collateral_ratio,certificate_type,ownership_match,estimated_dsr,eligibility_score,label,status_dhn,dhn_alasan,slik_n_loans,slik_worst_collectability,slik_n_banks,slik_total_outstanding,slik_total_installment_other,slik_avg_tenor_month,slik_has_macet,slik_has_credit_history,bank_n_accounts,bank_best_avg_balance_6m,bank_total_avg_credit,bank_total_avg_debit,bank_total_overdraft_6m,bank_any_dormant,net_profit_2024,net_profit_2025,operating_cashflow_2024,operating_cashflow_2025,revenue_2024,revenue_2025,total_asset_2024,total_asset_2025,total_liability_2024,total_liability_2025,revenue_growth_pct,profit_margin_2025,liability_to_asset_2025,agunan_kelurahan,land_area_m2,building_area_m2,nilai_tanah_per_m2,nilai_bangunan_per_m2,dsr_capped,is_female_owner,has_dhn_flag
0,APP202600001,3276010601750001,CIF1000001,2025-07-08,UMKM,UD Santoso Abadi,UD,Budi Panjaitan,L,51,Menikah,S2,Jawa Barat,Depok,Sukmajaya,Region 2,KCP Bogor Baranangsiang,Manufaktur,Konveksi,14,9,2672137,66,300000000,Rumah,"Poris Plawad, Tangerang",Banten,Tangerang,423.4,2328496000,1862796800,7.76,HGB,Ya,3.0,0.804,Diterima,Tidak,Tidak Berlaku,1.0,1.0,1.0,87078357.0,2882804.0,36.0,0.0,1,2,1002986,4408782,3829344,0,0,3405584.0,3318494.0,3039455.0,4499938.0,29666491.0,32065651.0,48148218.0,37084511.0,27015813.0,18787312.0,0.0809,0.1035,0.5066,Poris Plawad,187.3,236.1,8020000,3500000,3.0,0,0
1,APP202600002,3172010301920002,CIF1000002,2025-09-01,UMKM,UD Wijaya Mandiri,UD,Andi Hidayat,L,34,Cerai Hidup,S2,DKI Jakarta,Jakarta Utara,Kramat Jati,Region 1,KCP Bekasi Barat,Jasa,Bengkel,1,3,1807917,39,75000000,Rumah,"Pluit, Jakarta Utara",DKI Jakarta,Jakarta Utara,174.8,3724038000,2979230400,49.65,HGB,Ya,3.0,0.683,Diterima,Tidak,Tidak Berlaku,2.0,2.0,2.0,219070769.0,16155953.0,24.0,0.0,1,1,211129,1124770,1040039,0,0,2907631.0,2602253.0,3572130.0,2863796.0,22347316.0,21695009.0,27256315.0,35323008.0,17484128.0,9380894.0,-0.0292,0.1199,0.2656,Pluit,113.0,61.8,29970000,5460000,3.0,0,0
2,APP202600003,3671010604800003,CIF1000003,2025-10-27,UMKM,UD Kusuma Sejahtera,CV,Doni Pratama,L,46,Cerai Hidup,S2,Banten,Tangerang,Bekasi Timur,Region 3,KCP Cibubur,Transportasi,Ekspedisi Kecil,17,33,1698736,59,150000000,Tanah,"Sukabumi Selatan, Jakarta Barat",DKI Jakarta,Jakarta Barat,67.0,1681700000,1345360000,11.21,HGB,Ya,3.0,0.435,Ditolak,Tidak,Tidak Berlaku,2.0,4.0,2.0,135687676.0,15096128.0,24.0,0.0,1,1,851330,1394768,1222864,1,1,5615079.0,3670504.0,7404291.0,4126254.0,23914503.0,20384842.0,34076107.0,37104928.0,17322819.0,11862908.0,-0.1476,0.1801,0.3197,Sukabumi Selatan,67.0,0.0,25100000,5500000,3.0,0,0


## 3. Exploratory Data Analysis (EDA)

Semua visualisasi di bawah interaktif (hover untuk detail, bisa di-zoom).

### 3.1 Distribusi Label (Target)

In [5]:
label_counts = master["label"].value_counts().reset_index()
label_counts.columns = ["label", "jumlah"]

fig = px.pie(label_counts, names="label", values="jumlah", hole=0.45,
             color="label", color_discrete_map={"Diterima": "#2ecc71", "Ditolak": "#e74c3c"},
             title="Distribusi Keputusan Kredit (Label)")
fig.update_traces(textinfo="percent+label")
fig.show()


### 3.2 Kolektibilitas SLIK vs Keputusan Kredit
Semakin buruk kolektibilitas (Kurang Lancar → Macet), semakin tinggi proporsi
`Ditolak` — sesuai ekspektasi karena Character adalah komponen bobot terbesar
dalam skor kelayakan.

In [6]:
collect_label_map = {0: "Belum Ada Riwayat", 1: "Lancar", 2: "DPK",
                      3: "Kurang Lancar", 4: "Diragukan", 5: "Macet"}
master["collectability_readable"] = master["slik_worst_collectability"].map(collect_label_map)

ct = pd.crosstab(master["collectability_readable"], master["label"], normalize="index") * 100
ct = ct.reindex(["Belum Ada Riwayat","Lancar","DPK","Kurang Lancar","Diragukan","Macet"])
ct = ct.reset_index().melt(id_vars="collectability_readable", var_name="label", value_name="persen")

fig = px.bar(ct, x="collectability_readable", y="persen", color="label", barmode="stack",
             color_discrete_map={"Diterima": "#2ecc71", "Ditolak": "#e74c3c"},
             title="Kolektibilitas SLIK Terburuk vs Keputusan Kredit (%)",
             labels={"collectability_readable": "Kolektibilitas SLIK Terburuk", "persen": "Persentase (%)"})
fig.show()


### 3.3 DSR (Debt Service Ratio) vs Keputusan Kredit

In [7]:
fig = px.box(master, x="label", y="estimated_dsr", color="label",
             color_discrete_map={"Diterima": "#2ecc71", "Ditolak": "#e74c3c"},
             points="outliers",
             title="Distribusi DSR (Debt Service Ratio) berdasarkan Keputusan Kredit",
             labels={"estimated_dsr": "Estimasi DSR", "label": "Keputusan"})
fig.show()


### 3.4 Collateral Ratio (LTV) vs Keputusan Kredit

In [8]:
fig = px.box(master, x="label", y="collateral_ratio", color="label",
             color_discrete_map={"Diterima": "#2ecc71", "Ditolak": "#e74c3c"},
             points="outliers", log_y=True,
             title="Rasio Nilai Agunan terhadap Pinjaman (LTV) berdasarkan Keputusan Kredit",
             labels={"collateral_ratio": "Collateral Ratio (log scale)", "label": "Keputusan"})
fig.show()


### 3.5 Sebaran Sektor Industri vs Keputusan Kredit

In [9]:
ind_ct = master.groupby(["industry", "label"]).size().reset_index(name="jumlah")

fig = px.bar(ind_ct, x="industry", y="jumlah", color="label", barmode="group",
             color_discrete_map={"Diterima": "#2ecc71", "Ditolak": "#e74c3c"},
             title="Jumlah Pengajuan per Sektor Industri berdasarkan Keputusan",
             labels={"industry": "Sektor Industri", "jumlah": "Jumlah Pengajuan"})
fig.show()


### 3.6 Pertumbuhan Omset (2024→2025) vs Keputusan Kredit

In [10]:
fig = px.histogram(master, x="revenue_growth_pct", color="label", barmode="overlay",
                    nbins=50, opacity=0.65,
                    color_discrete_map={"Diterima": "#2ecc71", "Ditolak": "#e74c3c"},
                    title="Distribusi Pertumbuhan Omset 2024→2025 berdasarkan Keputusan Kredit",
                    labels={"revenue_growth_pct": "Pertumbuhan Omset (%)"})
fig.add_vline(x=0, line_dash="dash", line_color="gray")
fig.show()


### 3.7 Korelasi Antar Fitur Numerik

In [11]:
numeric_cols = ["owner_age","business_age_year","employee_count","monthly_turnover_est",
                "loan_requested","collateral_ratio","collateral_size_m2","estimated_dsr",
                "slik_worst_collectability","slik_n_loans","revenue_growth_pct",
                "profit_margin_2025","liability_to_asset_2025","bank_best_avg_balance_6m",
                "bank_total_overdraft_6m"]
corr = master[numeric_cols].corr().round(2)

fig = px.imshow(corr, text_auto=True, aspect="auto", color_continuous_scale="RdBu_r",
                 zmin=-1, zmax=1, title="Korelasi Antar Fitur Numerik Utama")
fig.update_layout(height=650)
fig.show()


### 3.8 Status DHN vs Keputusan Kredit

In [12]:
dhn_ct = master.groupby(["status_dhn", "label"]).size().reset_index(name="jumlah")

fig = px.bar(dhn_ct, x="status_dhn", y="jumlah", color="label", barmode="group",
             color_discrete_map={"Diterima": "#2ecc71", "Ditolak": "#e74c3c"},
             title="Status Daftar Hitam Nasional (DHN) vs Keputusan Kredit",
             labels={"status_dhn": "Terdaftar DHN?", "jumlah": "Jumlah Pengajuan"})
fig.show()


### 3.9 Sebaran Geografis Pengajuan

In [13]:
geo_ct = master.groupby(["province", "label"]).size().reset_index(name="jumlah")

fig = px.bar(geo_ct, x="province", y="jumlah", color="label", barmode="stack",
             color_discrete_map={"Diterima": "#2ecc71", "Ditolak": "#e74c3c"},
             title="Sebaran Pengajuan per Provinsi berdasarkan Keputusan Kredit",
             labels={"province": "Provinsi", "jumlah": "Jumlah Pengajuan"})
fig.show()


### 3.10 Nilai Agunan vs Nominal Pinjaman Diajukan

In [14]:
fig = px.scatter(master, x="loan_requested", y="collateral_market_value", color="label",
                  color_discrete_map={"Diterima": "#2ecc71", "Ditolak": "#e74c3c"},
                  hover_data=["company_name", "industry", "collateral_ratio"],
                  opacity=0.6,
                  title="Nilai Agunan vs Nominal Pinjaman Diajukan",
                  labels={"loan_requested": "Pinjaman Diajukan (IDR)",
                          "collateral_market_value": "Nilai Pasar Agunan (IDR)"})
# garis referensi LTV = 100% (agunan = pinjaman)
max_val = max(master["loan_requested"].max(), master["collateral_market_value"].quantile(0.98))
fig.add_shape(type="line", x0=0, y0=0, x1=max_val, y1=max_val,
              line=dict(color="gray", dash="dash"))
fig.show()


## 4. Agentic Screening Pipeline

Implementasi 7 sub-agent sesuai desain arsitektur (rule-based dulu, bisa
dikembangkan lebih lanjut pakai LLM untuk narasi yang lebih natural):

| Agent | Tugas | Terkait 5C |
|---|---|---|
| Identity Agent | Validasi Dukcapil (NIK & usia) | Prasyarat |
| Credit History Agent | Analisa SLIK | Character |
| DHN Agent | Cek daftar hitam | Character |
| Collateral Agent | Validasi ATR/BPN & LTV | Collateral |
| Financial Agent | Analisis laporan keuangan | Capacity |
| Cashflow Agent | Analisis mutasi rekening | Capacity |
| **Risk Agent** | Orkestrator — gabungkan semua hasil + hard rules + generate keputusan & narasi | Semua |

**Risk Agent** menerapkan *hard rules* (kill-switch) untuk kasus yang jelas
tidak layak (identitas tidak valid, masuk DHN, riwayat Macet) sebelum masuk ke
skema skor gabungan berbobot — meniru praktik underwriting bank sungguhan.

In [15]:
# =========================================================================
# AGENT 1: IDENTITY AGENT (validasi Dukcapil)
# =========================================================================
def identity_agent(row):
    valid_nik = len(str(row["NIK"])) == 16
    valid_age = row["owner_age"] >= 21
    passed = valid_nik and valid_age
    notes = []
    if not valid_nik: notes.append("Format NIK tidak valid")
    if not valid_age: notes.append("Usia pemohon di bawah 21 tahun")
    return pd.Series({
        "identity_passed": passed,
        "identity_notes": "; ".join(notes) if notes else "Identitas valid",
    })

# =========================================================================
# AGENT 2: CREDIT HISTORY AGENT (SLIK + OTS/wawancara -> Character)
# =========================================================================
def credit_history_agent(row):
    if row["slik_has_credit_history"] == 0:
        score = 0.6  # netral: belum ada rekam jejak, bukan positif/negatif
        notes = "Belum memiliki riwayat kredit di SLIK (nasabah baru)"
    else:
        collect = row["slik_worst_collectability"]
        score = {1: 1.0, 2: 0.75, 3: 0.45, 4: 0.2, 5: 0.0}.get(int(collect), 0.5)
        label_map = {1:"Lancar",2:"Dalam Perhatian Khusus",3:"Kurang Lancar",4:"Diragukan",5:"Macet"}
        notes = f"Kolektibilitas terburuk: {label_map.get(int(collect))} di {int(row['slik_n_banks'])} bank"
        if row["slik_has_macet"] == 1:
            notes += " — riwayat Macet ditemukan"
    return pd.Series({"character_score": round(score, 3), "character_notes": notes})

# =========================================================================
# AGENT 3: DHN AGENT
# =========================================================================
def dhn_agent(row):
    blacklisted = row["status_dhn"] == "Ya"
    notes = row["dhn_alasan"] if blacklisted else "Tidak terdaftar di Daftar Hitam Nasional"
    return pd.Series({"dhn_blacklisted": blacklisted, "dhn_notes": notes})

# =========================================================================
# AGENT 4: COLLATERAL AGENT (validasi ATR/BPN + LTV)
# =========================================================================
def collateral_agent(row):
    ratio = row["collateral_ratio"]
    match = row["ownership_match"] == "Ya"
    score = np.clip(ratio / 1.5, 0, 1)
    if not match:
        score = min(score, 0.2)
    notes = f"LTV agunan {ratio*100:.0f}% dari pinjaman"
    if not match:
        notes += " — nama sertifikat TIDAK sesuai pemilik (butuh verifikasi manual)"
    return pd.Series({"collateral_score": round(float(score), 3), "collateral_notes": notes})

# =========================================================================
# AGENT 5: FINANCIAL AGENT (laporan keuangan)
# =========================================================================
def financial_agent(row):
    growth = row["revenue_growth_pct"]
    margin = row["profit_margin_2025"]
    score = np.clip(0.5 + growth, 0, 1) * 0.6 + np.clip(margin / 0.15, 0, 1) * 0.4
    trend = "tumbuh" if growth > 0.02 else ("stagnan" if growth > -0.02 else "menurun")
    notes = f"Omset {trend} {growth*100:+.1f}% (2024→2025), margin laba {margin*100:.1f}%"
    return pd.Series({"financial_score": round(float(np.clip(score,0,1)), 3), "financial_notes": notes})

# =========================================================================
# AGENT 6: CASHFLOW AGENT (mutasi rekening)
# =========================================================================
def cashflow_agent(row):
    monthly_turnover = row["monthly_turnover_est"]
    balance_ratio = row["bank_best_avg_balance_6m"] / max(monthly_turnover, 1)
    overdraft_penalty = min(row["bank_total_overdraft_6m"] * 0.1, 0.3)
    score = np.clip(balance_ratio, 0, 1) - overdraft_penalty
    notes = f"Saldo rata-rata {balance_ratio*100:.0f}% dari omset bulanan"
    if row["bank_total_overdraft_6m"] > 0:
        notes += f", overdraft {int(row['bank_total_overdraft_6m'])}x dalam 6 bulan"
    if row["bank_any_dormant"] == 1:
        notes += ", memiliki rekening dormant"
    return pd.Series({"cashflow_score": round(float(np.clip(score,0,1)), 3), "cashflow_notes": notes})

# =========================================================================
# AGENT 7: RISK AGENT (orkestrator - gabungkan semua hasil di atas)
# =========================================================================
INDUSTRY_RISK_PENALTY = {"Perdagangan":0.02,"Kuliner":0.04,"Jasa":0.02,
                          "Manufaktur":0.03,"Pertanian":0.06,"Transportasi":0.05}
INTEREST_BY_ZONE = {"Hijau": 9.5, "Kuning": 12.0, "Merah": 15.0}
TENOR_BY_LOAN = {"KMK": 12, "KI": 36, "KPR": 120, "KKB": 48, "KK": 24}

def risk_agent(row):
    # --- Hard rules (kill-switch, override skor gabungan) ---
    if row["identity_passed"] == False:
        return pd.Series({
            "decision": "Tidak Layak", "zone": "Merah",
            "jenis_kredit_rekomendasi": "-", "nominal_disetujui": 0,
            "jangka_waktu_bulan": 0, "bunga_persen": None,
            "insight": f"Tidak layak karena {row['identity_notes'].lower()}.",
        })
    if row["dhn_blacklisted"]:
        return pd.Series({
            "decision": "Tidak Layak", "zone": "Merah",
            "jenis_kredit_rekomendasi": "-", "nominal_disetujui": 0,
            "jangka_waktu_bulan": 0, "bunga_persen": None,
            "insight": f"Tidak layak karena nasabah terdaftar di Daftar Hitam Nasional ({row['dhn_notes']}).",
        })
    if row["character_score"] == 0.0:  # riwayat Macet
        return pd.Series({
            "decision": "Tidak Layak", "zone": "Merah",
            "jenis_kredit_rekomendasi": "-", "nominal_disetujui": 0,
            "jangka_waktu_bulan": 0, "bunga_persen": None,
            "insight": "Tidak layak karena memiliki riwayat kredit Macet pada SLIK.",
        })

    # --- Weighted score (kalau lolos semua hard rule) ---
    industry_penalty = INDUSTRY_RISK_PENALTY.get(row["industry"], 0.03)
    condition_score = np.clip(1 - industry_penalty * 4, 0, 1)
    score = (0.35 * row["character_score"] + 0.25 * row["financial_score"]
             + 0.20 * row["collateral_score"] + 0.10 * row["cashflow_score"]
             + 0.10 * condition_score)

    if score >= 0.70:
        decision, zone = "Layak", "Hijau"
    elif score >= 0.55:
        decision, zone = "Layak Bersyarat", "Kuning"
    elif score >= 0.40:
        decision, zone = "Perlu Review Ulang", "Kuning"
    else:
        decision, zone = "Tidak Layak", "Merah"

    # nominal disetujui: ambil yg lebih kecil antara plafon diajukan & (LTV agunan / kebijakan haircut)
    max_by_collateral = row["collateral_market_value"] * 0.7  # bank biasa kasih maks 70% nilai agunan
    nominal = min(row["loan_requested"], max_by_collateral) if decision != "Tidak Layak" else 0
    nominal = int(nominal) if decision != "Tidak Layak" else 0
    jenis = "KMK" if row["loan_requested"] < 200_000_000 else "KI"  # modal kerja vs investasi, berdasar skala pinjaman
    tenor = TENOR_BY_LOAN.get(jenis, 24)
    bunga = INTEREST_BY_ZONE[zone] if decision != "Tidak Layak" else None

    # --- Narasi insight, kategori sesuai spesifikasi ---
    if decision == "Layak":
        insight = (f"Layak karena Character {row['character_score']:.2f}, "
                   f"Financial {row['financial_score']:.2f}, dan Collateral {row['collateral_score']:.2f} "
                   f"semuanya berada di zona aman.")
    elif decision == "Layak Bersyarat":
        weak_points = []
        if row["financial_score"] < 0.5: weak_points.append("kondisi keuangan cenderung lemah")
        if row["collateral_score"] < 0.5: weak_points.append("nilai agunan relatif pas-pasan")
        if row["cashflow_score"] < 0.5: weak_points.append("arus kas kurang stabil")
        alasan = ", ".join(weak_points) if weak_points else "beberapa indikator berada di batas ambang"
        insight = f"Layak bersyarat karena {alasan} — disarankan tambahan agunan/penjamin atau plafon diturunkan."
    elif decision == "Perlu Review Ulang":
        insight = (f"Perlu review ulang karena skor gabungan ({score:.2f}) berada di area abu-abu — "
                   f"disarankan OTS/wawancara lanjutan sebelum keputusan final.")
    else:
        insight = f"Tidak layak karena skor gabungan ({score:.2f}) di bawah ambang batas kelayakan."

    return pd.Series({
        "decision": decision, "zone": zone,
        "jenis_kredit_rekomendasi": jenis,
        "nominal_disetujui": nominal,
        "jangka_waktu_bulan": tenor if decision != "Tidak Layak" else 0,
        "bunga_persen": bunga,
        "insight": insight,
        "risk_score": round(float(score), 3),
    })

In [16]:
print("Menjalankan Identity Agent...")
master = master.join(master.apply(identity_agent, axis=1))
print("Menjalankan Credit History Agent...")
master = master.join(master.apply(credit_history_agent, axis=1))
print("Menjalankan DHN Agent...")
master = master.join(master.apply(dhn_agent, axis=1))
print("Menjalankan Collateral Agent...")
master = master.join(master.apply(collateral_agent, axis=1))
print("Menjalankan Financial Agent...")
master = master.join(master.apply(financial_agent, axis=1))
print("Menjalankan Cashflow Agent...")
master = master.join(master.apply(cashflow_agent, axis=1))
print("Menjalankan Risk Agent (orkestrator)...")
master = master.join(master.apply(risk_agent, axis=1))

print("\nDistribusi keputusan Agentic Pipeline:")
print(master["decision"].value_counts(normalize=True).round(3))

master[["application_id","company_name","decision","zone","jenis_kredit_rekomendasi",
        "nominal_disetujui","jangka_waktu_bulan","bunga_persen","insight"]].head(5)


Menjalankan Identity Agent...
Menjalankan Credit History Agent...
Menjalankan DHN Agent...
Menjalankan Collateral Agent...
Menjalankan Financial Agent...
Menjalankan Cashflow Agent...
Menjalankan Risk Agent (orkestrator)...

Distribusi keputusan Agentic Pipeline:
decision
Layak                 0.709
Layak Bersyarat       0.191
Tidak Layak           0.074
Perlu Review Ulang    0.025
Name: proportion, dtype: float64


,application_id,company_name,decision,zone,jenis_kredit_rekomendasi,nominal_disetujui,jangka_waktu_bulan,bunga_persen,insight
0,APP202600001,UD Santoso Abadi,Layak,Hijau,KI,300000000,36,9.5,"Layak karena Character 1.00, Financial 0.62, d..."
1,APP202600002,UD Wijaya Mandiri,Layak,Hijau,KMK,75000000,12,9.5,"Layak karena Character 0.75, Financial 0.60, d..."
2,APP202600003,UD Kusuma Sejahtera,Perlu Review Ulang,Kuning,KMK,150000000,12,12.0,Perlu review ulang karena skor gabungan (0.54)...
3,APP202600004,UD Susanto Makmur,Layak,Hijau,KI,200000000,36,9.5,"Layak karena Character 0.75, Financial 0.54, d..."
4,APP202600005,CV Wijaya Sejahtera,Layak,Hijau,KI,750000000,36,9.5,"Layak karena Character 0.75, Financial 0.87, d..."


## 5. Validasi Pipeline vs Label Asli

Ini bukan evaluasi model ML (karena pipeline-nya rule-based, bukan trained
model) — tapi sanity check untuk lihat seberapa dekat logika agent dengan
keputusan yang dipakai untuk generate label ground truth.

In [17]:
cm = pd.crosstab(master["decision"], master["label"])
cm = cm.reindex(["Layak", "Layak Bersyarat", "Perlu Review Ulang", "Tidak Layak"])

fig = px.imshow(cm, text_auto=True, aspect="auto", color_continuous_scale="Blues",
                 title="Keputusan Agentic Pipeline vs Label Asli (Ground Truth)",
                 labels={"x": "Label Asli (dari generator)", "y": "Keputusan Agent Pipeline",
                         "color": "Jumlah"})
fig.show()

agree_rate = (master["decision"].isin(["Layak","Layak Bersyarat"]) == (master["label"]=="Diterima")).mean()
print(f"Tingkat kesesuaian keputusan agent pipeline vs label asli: {agree_rate:.1%}")
print("\nDistribusi zona risiko:")
print(master["zone"].value_counts(normalize=True).round(3))


Tingkat kesesuaian keputusan agent pipeline vs label asli: 92.7%

Distribusi zona risiko:
zone
Hijau     0.709
Kuning    0.216
Merah     0.074
Name: proportion, dtype: float64


## 6. Export Dataset Hasil Scoring (untuk Streamlit Dashboard)

In [18]:
export_cols = ["application_id","NIK","company_name","owner_name","industry",
    "sub_industry","province","city","branch_name","region",
    "loan_requested","collateral_type","collateral_market_value","collateral_ratio",
    "estimated_dsr","revenue_growth_pct","profit_margin_2025",
    "slik_worst_collectability","status_dhn",
    "identity_passed","character_score","character_notes",
    "collateral_score","collateral_notes","financial_score","financial_notes",
    "cashflow_score","cashflow_notes","risk_score",
    "decision","zone","jenis_kredit_rekomendasi","nominal_disetujui",
    "jangka_waktu_bulan","bunga_persen","insight","label"]

master_export = master[export_cols].copy()
out_path = f"{OUT_DIR}/master_scored.csv"
master_export.to_csv(out_path, index=False)
print(f"Tersimpan: {out_path}  ({master_export.shape[0]} baris x {master_export.shape[1]} kolom)")
master_export.head(3)


Tersimpan: ./dataset/master_scored.csv  (3000 baris x 37 kolom)


,application_id,NIK,company_name,owner_name,industry,sub_industry,province,city,branch_name,region,loan_requested,collateral_type,collateral_market_value,collateral_ratio,estimated_dsr,revenue_growth_pct,profit_margin_2025,slik_worst_collectability,status_dhn,identity_passed,character_score,character_notes,collateral_score,collateral_notes,financial_score,financial_notes,cashflow_score,cashflow_notes,risk_score,decision,zone,jenis_kredit_rekomendasi,nominal_disetujui,jangka_waktu_bulan,bunga_persen,insight,label
0,APP202600001,3276010601750001,UD Santoso Abadi,Budi Panjaitan,Manufaktur,Konveksi,Jawa Barat,Depok,KCP Bogor Baranangsiang,Region 2,300000000,Rumah,2328496000,7.76,3.0,0.0809,0.1035,1.0,Tidak,True,1.00,Kolektibilitas terburuk: Lancar di 1 bank,1.0,LTV agunan 776% dari pinjaman,0.625,"Omset tumbuh +8.1% (2024→2025), margin laba 10.3%",0.375,Saldo rata-rata 38% dari omset bulanan,0.832,Layak,Hijau,KI,300000000,36,9.5,"Layak karena Character 1.00, Financial 0.62, d...",Diterima
1,APP202600002,3172010301920002,UD Wijaya Mandiri,Andi Hidayat,Jasa,Bengkel,DKI Jakarta,Jakarta Utara,KCP Bekasi Barat,Region 1,75000000,Rumah,3724038000,49.65,3.0,-0.0292,0.1199,2.0,Tidak,True,0.75,Kolektibilitas terburuk: Dalam Perhatian Khusu...,1.0,LTV agunan 4965% dari pinjaman,0.602,"Omset menurun -2.9% (2024→2025), margin laba 1...",0.117,Saldo rata-rata 12% dari omset bulanan,0.717,Layak,Hijau,KMK,75000000,12,9.5,"Layak karena Character 0.75, Financial 0.60, d...",Diterima
2,APP202600003,3671010604800003,UD Kusuma Sejahtera,Doni Pratama,Transportasi,Ekspedisi Kecil,Banten,Tangerang,KCP Cibubur,Region 3,150000000,Tanah,1681700000,11.21,3.0,-0.1476,0.1801,4.0,Tidak,True,0.20,Kolektibilitas terburuk: Diragukan di 2 bank,1.0,LTV agunan 1121% dari pinjaman,0.611,"Omset menurun -14.8% (2024→2025), margin laba ...",0.401,"Saldo rata-rata 50% dari omset bulanan, overdr...",0.543,Perlu Review Ulang,Kuning,KMK,150000000,12,12.0,Perlu review ulang karena skor gabungan (0.54)...,Ditolak


## 7. Ide Dashboard Streamlit

Beberapa ide halaman/komponen untuk dashboard Streamlit yang bisa dibangun
dari `master_scored.csv`:

**A. Halaman Overview (Beranda)**
- KPI card: total pengajuan, % Layak/Tidak Layak, rata-rata risk_score, total nominal disetujui
- Pie/bar chart distribusi zona risiko (Hijau/Kuning/Merah)
- Filter global: cabang, region, sektor industri, rentang tanggal pengajuan

**B. Halaman Daftar Pengajuan (Tabel Interaktif)**
- Tabel semua pengajuan dgn kolom: nama usaha, industri, nominal, decision, zone
- Bisa di-sort/filter per kolom (`st.dataframe` dengan `column_config`, atau AgGrid)
- Klik satu baris → drill-down ke halaman detail

**C. Halaman Detail Nasabah (Drill-down)**
- Tampilkan hasil dari **setiap agent** secara terpisah (mirip "kartu" per-agent):
  Identity ✅/❌, Character (skor + catatan SLIK/OTS), Collateral (skor + LTV),
  Financial (skor + tren omset), Cashflow (skor + saldo/overdraft)
- Insight/narasi akhir dari Risk Agent ditampilkan mencolok di atas (mirip "ringkasan eksekutif")
- Rekomendasi: jenis kredit, nominal, tenor, bunga

**D. Halaman Simulasi / Input Manual**
- Form input data nasabah baru (atau slider untuk ubah parameter nasabah existing)
- Jalankan ulang pipeline agent secara *live* saat form disubmit → tampilkan hasil real-time
- Berguna buat demo capstone: reviewer bisa coba-coba input sendiri

**E. Halaman Monitoring Portofolio**
- Breakdown risiko per cabang/region/sektor industri (heatmap atau treemap)
- Analisis: cabang mana yang approval rate-nya paling rendah/tinggi, kenapa
- Sekaligus jadi bekal untuk pengembangan lanjutan ke arah "Risk Agent monitoring" (SIMON) di masa depan

**Tips teknis:**
- `st.session_state` untuk simpan hasil filter/pilihan antar halaman
- `st.plotly_chart(fig, use_container_width=True)` supaya chart Plotly dari notebook ini bisa langsung dipakai lagi di Streamlit
- Multi-page app: folder `pages/` (`1_Overview.py`, `2_Daftar_Pengajuan.py`, dst)
- Cache load data dgn `@st.cache_data` biar ga reload CSV tiap interaksi